<a href="https://colab.research.google.com/github/Sosina47/aws-sbg-tasks/blob/main/spam_filter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split

In [ ]:
emails = [
    # Spam messages
    "Congratulations! You have won a free prize. Claim now!",
    "WINNER! Call this number to receive your cash reward.",
    "Get a free iPhone today. Limited offer!",
    "You have been selected for a special cash bonus.",
    "Claim your free gift before the offer expires.",
    "URGENT! Your account has won a lottery prize.",
    "Earn money from home. Register now!",
    "Click here to claim your reward.",
    "You have won 1000 dollars. Contact us now!",
    "Exclusive offer! Buy one get one free!",

    # Ham messages
    "Hey, are we meeting tomorrow?",
    "I'll call you when I arrive home.",
    "Don't forget to bring your notebook.",
    "Can you send me the assignment?",
    "Happy birthday! Have a wonderful day.",
    "I'm running late. Please wait for me.",
    "Are you coming to class today?",
    "Thanks for helping me with the project.",
    "Let's meet at the library this afternoon.",
    "Your package has arrived. You can collect it."
]

labels = [
    "spam", "spam", "spam", "spam", "spam",
    "spam", "spam", "spam", "spam", "spam",

    "ham", "ham", "ham", "ham", "ham",
    "ham", "ham", "ham", "ham", "ham"
]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    emails,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

In [ ]:
vec = TfidfVectorizer()

X_train_features = vec.fit_transform(X_train)

## 4. Training the Spam Classifier

In [ ]:
model = MultinomialNB()

model.fit(X_train_features, y_train)

MultinomialNB()

In [ ]:
print("Classes:", model.classes_)

print("\nClass log probabilities:")
print(model.class_log_prior_)

print("\nFeature log probabilities shape:")
print(model.feature_log_prob_.shape)

Classes: ['ham' 'spam']

Class log probabilities:
[-0.69314718 -0.69314718]

Feature log probabilities shape:
(2, 77)


In [ ]:
feature_names = vec.get_feature_names_out()

for i, class_name in enumerate(model.classes_):
    top_indices = model.feature_log_prob_[i].argsort()[-10:][::-1]

    print(f"\nTop features for {class_name}:")
    for index in top_indices:
        print(feature_names[index])


Top features for ham:
me
for
the
you
your
happy
wonderful
day
birthday
don

Top features for spam:
claim
your
won
free
have
you
prize
reward
now
one


In [9]:
model.fit(X_train_features, y_train)

MultinomialNB()

In [10]:
X_test_features = vec.transform(X_test)

In [11]:
predictions = model.predict(X_test_features)

for message, actual, predicted in zip(X_test, y_test, predictions):
    print("Message:", message)
    print("Actual:", actual)
    print("Predicted:", predicted)
    print("-" * 50)

Message: Are you coming to class today?
Actual: ham
Predicted: ham
--------------------------------------------------
Message: Can you send me the assignment?
Actual: ham
Predicted: ham
--------------------------------------------------
Message: Get a free iPhone today. Limited offer!
Actual: spam
Predicted: spam
--------------------------------------------------
Message: Earn money from home. Register now!
Actual: spam
Predicted: spam
--------------------------------------------------


In [12]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, predictions))

print("\nClassification Report:")
print(classification_report(y_test, predictions, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, predictions))

Accuracy: 1.0

Classification Report:
              precision    recall  f1-score   support

         ham       1.00      1.00      1.00         2
        spam       1.00      1.00      1.00         2

    accuracy                           1.00         4
   macro avg       1.00      1.00      1.00         4
weighted avg       1.00      1.00      1.00         4


Confusion Matrix:
[[2 0]
 [0 2]]


In [13]:
# Original spam message
original = "Congratulations! You won a free prize!"

# Obfuscated spam message
obfuscated = "C0ngratulati0ns! Y0u w0n a fr33 pr1ze!"

# Convert both messages using our existing vectorizer
original_features = vec.transform([original])
obfuscated_features = vec.transform([obfuscated])

# Predict their classes
original_prediction = model.predict(original_features)[0]
obfuscated_prediction = model.predict(obfuscated_features)[0]

print("Original message:", original)
print("Original prediction:", original_prediction)

print("\nObfuscated message:", obfuscated)
print("Obfuscated prediction:", obfuscated_prediction)

Original message: Congratulations! You won a free prize!
Original prediction: spam

Obfuscated message: C0ngratulati0ns! Y0u w0n a fr33 pr1ze!
Obfuscated prediction: ham


In [14]:
print("Original non-zero features:",
      original_features.nnz)

print("Obfuscated non-zero features:",
      obfuscated_features.nnz)

Original non-zero features: 5
Obfuscated non-zero features: 0


In [15]:
print("Original tokens recognized:")
print(vec.build_analyzer()(original))

print("\nObfuscated tokens recognized:")
print(vec.build_analyzer()(obfuscated))

Original tokens recognized:
['congratulations', 'you', 'won', 'free', 'prize']

Obfuscated tokens recognized:
['c0ngratulati0ns', 'y0u', 'w0n', 'fr33', 'pr1ze']


In [16]:
negation_tests = [
    "You have won a free prize!",
    "You have NOT won a free prize!",
    "You have not won anything. This is a scam warning."
]

features = vec.transform(negation_tests)
predictions = model.predict(features)

for message, prediction in zip(negation_tests, predictions):
    print(f"Message: {message}")
    print(f"Prediction: {prediction}")
    print("-" * 60)

Message: You have won a free prize!
Prediction: spam
------------------------------------------------------------
Message: You have NOT won a free prize!
Prediction: spam
------------------------------------------------------------
Message: You have not won anything. This is a scam warning.
Prediction: spam
------------------------------------------------------------


In [17]:
business_tests = [
    "Your free prize claim has been received by our office.",
    "Please review our special offer before tomorrow.",
    "Your account has been selected for a free reward.",
    "Thank you for contacting our customer service team.",
    "We received your request and will contact you tomorrow."
]

features = vec.transform(business_tests)
predictions = model.predict(features)

for message, prediction in zip(business_tests, predictions):
    print(f"Message: {message}")
    print(f"Prediction: {prediction}")
    print("-" * 60)

Message: Your free prize claim has been received by our office.
Prediction: spam
------------------------------------------------------------
Message: Please review our special offer before tomorrow.
Prediction: spam
------------------------------------------------------------
Message: Your account has been selected for a free reward.
Prediction: spam
------------------------------------------------------------
Message: Thank you for contacting our customer service team.
Prediction: ham
------------------------------------------------------------
Message: We received your request and will contact you tomorrow.
Prediction: ham
------------------------------------------------------------
